# ITRI626 – train all models on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When asked, upload **MLProject_reviewed.zip** (the latest zip from Claude, about 38 MB).
4. Wait for training to finish (about 1 hour). Keep this tab open: Colab disconnects tabs that are idle or closed.
5. At the end, `MLProject_results.zip` downloads by itself. Unzip it **into your MLProject folder** on your laptop and say yes to overwriting files.

The results zip contains the outputs, the notebook with all its outputs filled in, and the report tables and figures. It leaves out the large model weight files (`.pt`), which the brief says you don't have to submit.

### 1. Check that the GPU is on

In [ ]:
import torch, subprocess
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all again."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

### 2. Upload the project zip

In [ ]:
import os, zipfile, shutil, glob
from google.colab import files
os.chdir("/content")
shutil.rmtree("/content/MLProject", ignore_errors=True)   # remove any old copy from an earlier run
for old in glob.glob("/content/*.zip"):
    os.remove(old)
up = files.upload()                                        # choose MLProject_reviewed.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("/content")
assert os.path.isfile("/content/MLProject/src/mlproject/__main__.py"), "Wrong zip: use MLProject_reviewed.zip"
assert os.path.isfile("/content/MLProject/data/manual_exclude.csv"), "Old zip: use MLProject_reviewed.zip"
os.chdir("/content/MLProject")
print("raw images:", len(glob.glob("data/raw/*/*.jpg")), "(1,902 are used after review and cleaning)")

### 3. Install the few packages Colab does not already have

In [ ]:
!pip -q install seaborn tqdm nbconvert

### 4. Train and evaluate all three models (this is the long step)

In [ ]:
%env PYTHONPATH=/content/MLProject/src
!python -m mlproject run-all 2>&1 | grep -v -E "it/s\]|s/it\]" 

### 5. Run the project notebook so it is saved with all outputs
It reuses the checkpoints from step 4, so it only takes a few minutes.

In [ ]:
!cd notebooks && jupyter nbconvert --to notebook --execute microcontroller_classification.ipynb --inplace --ExecutePreprocessor.timeout=3600
print("notebook executed")

### 6. Quick look at the results

In [ ]:
import json, pandas as pd
rows = {}
for m in ["custom_cnn", "resnet18", "vit_b_16"]:
    r = json.load(open(f"outputs/results/{m}_results.json"))
    rows[m] = {k: r[k] for k in ["test_accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mean_latency_ms_per_image"]}
pd.DataFrame(rows).T

### 7. Download the results

In [ ]:
out = "/content/MLProject_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["outputs/**/*", "report/generated/*", "report/figures/*", "data/splits/*",
                    "notebooks/microcontroller_classification.ipynb"]:
        for f in glob.glob(pattern, recursive=True):
            if os.path.isfile(f) and not f.endswith(".pt"):
                z.write(f, f)
print(round(os.path.getsize(out) / 1e6, 1), "MB")
files.download(out)